# Pay for Data — Heurist Finance Agent

## Overview

A finance research agent that pays for real-time market data using **Amazon Bedrock AgentCore payments**. The agent calls paid [Heurist](https://heurist.xyz) endpoints for live prices, SEC filings, and macro indicators, analyzes the data with AgentCore Code Interpreter, and returns charts and reports as S3 presigned URLs — all without any manual payment code in the tools.

The agent is deployed to **AgentCore Runtime**: a managed container endpoint with HTTPS invocation, SigV4 auth, and automatic observability via CloudWatch.

### Use Case Details

| Information | Details |
|:---|:---|
| Use case type | Agentic data retrieval with autonomous micropayments |
| Agent type | Single |
| Payment protocol | x402 (HTTP 402 Payment Required) |
| Agentic framework | [Strands Agents](https://strandsagents.com/) |
| LLM model | Claude Sonnet 4 on Amazon Bedrock (configurable) |
| SDK used | `bedrock-agentcore[strands-agents]` (public PyPI) |
| Wallet type | Embedded crypto wallet (Coinbase CDP) |
| Payment network | Base mainnet (USDC) |

### Architecture

```
App Backend (ManagementRole)              AgentCore Runtime
  |                                        +------------------------------+
  | create_session(budget=$X)              |  runtime_agent.py            |
  |                                        |  BedrockAgentCoreApp         |
  |-- invoke(manager_arn, session_id, -->  |  + AgentCorePaymentsPlugin   |
  |         instrument_id, prompt)         |                              |
  |                                        |  http_request -> 402         |
  |<-- {response, artifacts: [{url}]} ---  |  -> ProcessPayment -> retry  |
  |                                        |  -> Code Interpreter         |
  | get_session(check spend)               |  -> export to S3             |
                                           +------------------------------+
                                                      |
                                                      v
                                          CloudWatch GenAI Observability
                                          (automatic via OpenTelemetry)
```

### AgentCore Capabilities Demonstrated

| Capability | How it is used here |
|:---|:---|
| **Payment manager** | Central resource that authorizes and tracks all payment activity. |
| **Payment instrument** | An embedded crypto wallet (Coinbase CDP, USDC on Base). |
| **Payment session** | A time-bounded, budget-capped authorization (`maxSpendAmount`). |
| **Payment processing** | End-to-end x402 negotiation, proof generation, retry, and on-chain settlement. |
| **AgentCore Runtime** | Managed container hosting with HTTPS endpoint and SigV4 auth. |
| **AgentCore Code Interpreter** | Remote sandboxed Python environment for pandas/matplotlib analysis. |
| **Observability** | Automatic OTel traces + logs in CloudWatch GenAI dashboard. |

### Notebook Flow

| Step | What happens |
|------|-------------|
| 1 | Configure credentials and confirm AWS identity |
| 2 | Sync the Heurist tool catalog (bundled in the container image) |
| 3 | Create the S3 artifacts bucket |
| 4 | Install the AgentCore CLI, scaffold and deploy |
| 5 | Add IAM permissions to the execution role |
| 6 | Invoke the deployed agent and inspect results |
| 7 | View observability traces in CloudWatch |
| 8 | Cleanup |

**Before running:**
1. `pip install -r requirements.txt`
2. `cp .env.example .env` and fill in your credentials (payment manager ARN, instrument ID, session ID)
3. Ensure Node.js 20+, Docker, and AWS CDK are installed

See [`README.md`](README.md) for full setup details.

## Install dependencies

In [ ]:
%pip install -r requirements.txt --quiet

## Step 1 — Configure credentials

Load credentials from `.env` and confirm AWS identity. Copy `.env.example` to `.env`
and fill in the values from the setup tutorial (`00-getting-started/`).

In [ ]:
import boto3
import json
from heurist_finance_agent.config import get_config

cfg = get_config()
print(f"Region:           {cfg.aws_region}")
print(f"Payment manager:  {cfg.payment_manager_arn}")
print(f"Payment session:  {cfg.payment_session_id}")
print(f"Payment instrument: {cfg.payment_instrument_id}")
print(f"Model:            {cfg.bedrock_model_id}")

session = boto3.Session()
identity = session.client('sts').get_caller_identity()
account_id = identity['Account']
region = cfg.aws_region
print(f"\nAuthenticated as: {identity['Arn']}")
print(f"Account:          {account_id}")

## Step 2 — Sync the Heurist tool catalog

Fetches the current registry of x402-enabled endpoints from the Heurist mesh and caches
it locally. The Runtime container image bundles this cache at build time — the deployed
agent reads the catalog without calling the Heurist registry at startup.

Re-run this cell before deploying to ensure the container has a current catalog.

In [ ]:
from heurist_finance_agent.catalog import fetch_live_catalog, get_tools_for_agents

catalog = fetch_live_catalog()
selected = get_tools_for_agents(cfg.heurist_tool_agent_ids)

print(f"Agents in registry: {catalog['count']}")
print(f"Selected agents:    {', '.join(cfg.heurist_tool_agent_ids)}")
print(f"Loaded paid tools:  {len(selected)}")
print()
for t in selected:
    print(f"  {t['agent_id']:30s}  {t['tool_name']:35s}  ${t['price_usd']:.3f}")

## Step 3 — Create the S3 artifacts bucket

Charts, reports, and CSVs produced by the agent in Code Interpreter are uploaded to S3.
The agent returns presigned download URLs in the invocation response — valid for
`CI_ARTIFACTS_TTL` seconds (default: 1 hour).

The bucket is private; no public access is granted. Skip this cell if you already have
a bucket — just set `ARTIFACTS_BUCKET` to its name.

In [ ]:
ARTIFACTS_BUCKET = f"heurist-finance-artifacts-{account_id}-{region}"

s3 = boto3.client('s3', region_name=region)
try:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=ARTIFACTS_BUCKET)
    else:
        s3.create_bucket(
            Bucket=ARTIFACTS_BUCKET,
            CreateBucketConfiguration={'LocationConstraint': region},
        )
    s3.put_public_access_block(
        Bucket=ARTIFACTS_BUCKET,
        PublicAccessBlockConfiguration={
            'BlockPublicAcls': True, 'IgnorePublicAcls': True,
            'BlockPublicPolicy': True, 'RestrictPublicBuckets': True,
        },
    )
    print(f'Created bucket: {ARTIFACTS_BUCKET}')
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f'Bucket already exists: {ARTIFACTS_BUCKET}')

print(f'Artifacts will be stored at: s3://{ARTIFACTS_BUCKET}/heurist-finance-artifacts/')

## Step 4 — Deploy to AgentCore Runtime

The `@aws/agentcore` CLI scaffolds a project, builds a Docker image, pushes it to ECR,
and deploys via CDK. First deploy takes ~3–5 minutes.

> **Prerequisites:** Node.js 20+, Docker running, AWS CDK installed
>
> **Cost notice:** This creates billable AWS resources (ECR, Runtime endpoint, CloudWatch logs).
> Run the cleanup section when finished.

In [ ]:
!npm install -g @aws/agentcore
!agentcore --version

In [ ]:
import os
import shutil

PROJECT_NAME = 'HeuristFinanceAgent'

# Scaffold
if not os.path.exists(PROJECT_NAME):
    !agentcore create --name {PROJECT_NAME} --framework Strands --protocol HTTP --model-provider Bedrock --memory none
else:
    print(f'{PROJECT_NAME}/ already exists — skipping create')

In [ ]:
dest_app = f'{PROJECT_NAME}/app/{PROJECT_NAME}'

# Copy runtime_agent.py as the entry point
shutil.copy('heurist_finance_agent/runtime_agent.py', f'{dest_app}/main.py')
print(f'Copied runtime_agent.py -> {dest_app}/main.py')

# Copy the heurist_finance_agent package (includes catalog cache)
pkg_dest = f'{dest_app}/heurist_finance_agent'
if os.path.exists(pkg_dest):
    shutil.rmtree(pkg_dest)
shutil.copytree('heurist_finance_agent', pkg_dest)
print(f'Copied heurist_finance_agent/ -> {pkg_dest}/')

cache_in_image = os.path.join(pkg_dest, 'catalog_live_cache.json')
if os.path.exists(cache_in_image):
    print('catalog_live_cache.json bundled in image')
else:
    print('WARNING: catalog_live_cache.json missing — re-run Step 2 first')

In [ ]:
# Write pyproject.toml — includes Code Interpreter
# (Code Interpreter is a remote AWS API; it works identically from Runtime containers)
pyproject_content = f'''[project]
name = "heurist-finance-agent"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "bedrock-agentcore[strands-agents]>=1.9.0",
    "boto3>=1.43.1",
    "strands-agents>=1.36.0",
    "strands-agents-tools[http_request,agent_core_code_interpreter]>=0.5.0",
    "requests>=2.32.0",
    "python-dotenv>=1.0.0",
]
'''

# Write .env — service config only, no payment credentials
runtime_env = f'''# Runtime config bundled in container image
# Payment credentials are NOT here — they come from the invocation payload
CI_ARTIFACTS_BUCKET={ARTIFACTS_BUCKET}
CI_ARTIFACTS_PREFIX=heurist-finance-artifacts
CI_ARTIFACTS_TTL=3600
AWS_REGION={region}
BEDROCK_MODEL_ID={cfg.bedrock_model_id}
'''

with open(f'{dest_app}/pyproject.toml', 'w') as f:
    f.write(pyproject_content)
with open(f'{dest_app}/.env', 'w') as f:
    f.write(runtime_env)

lock_file = f'{dest_app}/uv.lock'
if os.path.exists(lock_file):
    os.remove(lock_file)

print('pyproject.toml and .env written')
print(f'  CI_ARTIFACTS_BUCKET={ARTIFACTS_BUCKET}')
print(f'  BEDROCK_MODEL_ID={cfg.bedrock_model_id}')

In [ ]:
# Deploy (~3-5 min first time)
!cd {PROJECT_NAME} && agentcore deploy -y

In [ ]:
!cd {PROJECT_NAME} && agentcore status

## Step 5 — Add permissions to the execution role

The CLI auto-creates an execution role with Bedrock + CloudWatch permissions.
Add three more sets:

1. **Payment data-plane** — `ProcessPayment` and read operations
2. **Code Interpreter** — `StartCodeInterpreterSession`, `InvokeCodeInterpreter`, `StopCodeInterpreterSession`
3. **S3 artifacts** — `PutObject` + `GetObject` scoped to the artifacts bucket

The execution role cannot create payment sessions or instruments — that stays with the app backend (ManagementRole).

In [ ]:
iam = boto3.client('iam')

# Find the execution role created by agentcore deploy
paginator = iam.get_paginator('list_roles')
runtime_roles = []
for page in paginator.paginate():
    for role in page['Roles']:
        name = role['RoleName']
        if PROJECT_NAME.lower().replace('-', '') in name.lower().replace('-', '') and \
           ('execution' in name.lower() or 'exec' in name.lower()):
            runtime_roles.append(name)

if not runtime_roles:
    for page in paginator.paginate():
        for role in page['Roles']:
            if PROJECT_NAME.lower() in role['RoleName'].lower():
                runtime_roles.append(role['RoleName'])

assert runtime_roles, f'No {PROJECT_NAME} execution role found. Check agentcore deploy output.'
RUNTIME_ROLE_NAME = runtime_roles[0]
print(f'Execution role: {RUNTIME_ROLE_NAME}')

# 1. Payment data-plane
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName='PaymentDataPlaneAccess',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'PaymentDataPlaneAccess',
            'Effect': 'Allow',
            'Action': [
                'bedrock-agentcore:ProcessPayment',
                'bedrock-agentcore:GetPaymentInstrument',
                'bedrock-agentcore:GetPaymentInstrumentBalance',
                'bedrock-agentcore:GetPaymentSession',
                'bedrock-agentcore:GetResourcePaymentToken',
            ],
            'Resource': f'arn:aws:bedrock-agentcore:{region}:{account_id}:payment-manager/*',
        }],
    }),
)
print('Payment data-plane permissions added')

# 2. Code Interpreter
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName='CodeInterpreterAccess',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'CodeInterpreterAccess',
            'Effect': 'Allow',
            'Action': [
                'bedrock-agentcore:StartCodeInterpreterSession',
                'bedrock-agentcore:StopCodeInterpreterSession',
                'bedrock-agentcore:InvokeCodeInterpreter',
            ],
            'Resource': f'arn:aws:bedrock-agentcore:{region}:{account_id}:code-interpreter/*',
        }],
    }),
)
print('Code Interpreter permissions added')

# 3. S3 artifacts
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName='S3ArtifactsAccess',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'S3ArtifactsReadWrite',
            'Effect': 'Allow',
            'Action': ['s3:PutObject', 's3:GetObject'],
            'Resource': f'arn:aws:s3:::{ARTIFACTS_BUCKET}/heurist-finance-artifacts/*',
        }],
    }),
)
print(f'S3 artifact permissions added (bucket: {ARTIFACTS_BUCKET})')

print('\nPermissions summary:')
print('  Payment: ProcessPayment, GetPaymentInstrument, GetPaymentSession')
print('  Code Interpreter: StartSession, StopSession, InvokeCodeInterpreter')
print('  S3: PutObject + GetObject (artifacts bucket prefix only)')
print('  Not granted: CreateSession, CreateInstrument (stays with ManagementRole)')

## Step 6 — Invoke the deployed agent

Create a fresh payment session (the app backend controls the budget), then invoke
the deployed agent. The response includes the research summary and presigned S3 URLs
for any charts or reports the agent produced.

```json
{
  "response": "<markdown research summary>",
  "artifacts": [
    {"name": "chart.png", "url": "https://...", "expires_in": 3600}
  ]
}
```

In [ ]:
from bedrock_agentcore.payments import PaymentManager

manager = PaymentManager(payment_manager_arn=cfg.payment_manager_arn, region_name=region)

invoke_session = manager.create_payment_session(
    user_id=cfg.user_id,
    limits={'maxSpendAmount': {'value': '0.25', 'currency': 'USD'}},
    expiry_time_in_minutes=60,
)
session_id = invoke_session['paymentSessionId']
print(f'Payment session: {session_id} (budget: $0.25, expiry: 60 min)')

In [ ]:
invoke_payload = json.dumps({
    'prompt': (
        'Use FredMacroAgent to fetch the latest US GDP growth rate and unemployment rate. '
        'Use Code Interpreter to create a bar chart comparing them and a markdown summary. '
        'Save both as artifacts.'
    ),
    'payment_manager_arn': cfg.payment_manager_arn,
    'user_id': cfg.user_id,
    'payment_session_id': session_id,
    'payment_instrument_id': cfg.payment_instrument_id,
})

print(f'Invoking {PROJECT_NAME}...')
!cd {PROJECT_NAME} && agentcore invoke '{invoke_payload}'

In [ ]:
# Check session spend
session_info = manager.get_payment_session(
    user_id=cfg.user_id,
    payment_session_id=session_id,
)

available = session_info.get('availableLimits', {}).get('availableSpendAmount', {})
budget = session_info.get('limits', {}).get('maxSpendAmount', {})
budget_val = float(budget.get('value', 0))
avail_val = float(available.get('value', 0)) if available.get('value') else budget_val
spent = budget_val - avail_val

print(f'Session spend summary:')
print(f'  Budget:    ${budget_val:.4f} {budget.get("currency", "USD")}')
print(f'  Remaining: ${avail_val:.4f} {available.get("currency", "USD")}')
print(f'  Spent:     ${spent:.4f} USD')

## Step 7 — Observability

AgentCore Runtime automatically instruments the container with OpenTelemetry. Traces
and logs appear in **CloudWatch GenAI Observability** immediately after the first
invocation — no additional setup required.

Each invocation produces spans for:
- **LLM calls** — model ID, token counts, latency
- **Tool calls** — `http_request` invocations including x402 retry attempts
- **Agent turns** — full prompt → tool use → response cycle

The `AgentCorePaymentsPlugin` appears as child spans under the `http_request` spans,
showing the 402 intercept, `ProcessPayment` call, and retry.

> **Note:** Payment manager vended logs (`ProcessPayment`, `CreateSession` events)
> are configured separately via `enable_observability()` in `utils.py`. See Tutorial 00
> Step 8a for that setup.

In [ ]:
print('CloudWatch GenAI Observability dashboard:')
print(f'  https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#gen-ai-observability/agent-core')
print()
print('Stream live logs:')
print(f'  cd {PROJECT_NAME} && agentcore logs')

## Step 8 — Cleanup

> ⚠️ The following cells permanently delete the Runtime deployment and the S3 artifacts
> bucket. Download any artifacts you want to keep before proceeding.

In [ ]:
# Remove the AgentCore Runtime stack (ECR image, CloudWatch logs, CDK stack)
!cd {PROJECT_NAME} && agentcore remove all -y

In [ ]:
# Remove the scaffolded project directory
import shutil
if os.path.exists(PROJECT_NAME):
    shutil.rmtree(PROJECT_NAME)
    print(f'Removed {PROJECT_NAME}/')
else:
    print(f'{PROJECT_NAME}/ already removed')

# Empty and delete the S3 artifacts bucket
import boto3
s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket(ARTIFACTS_BUCKET)
try:
    bucket.objects.all().delete()
    bucket.delete()
    print(f'Deleted S3 bucket: {ARTIFACTS_BUCKET}')
except Exception as e:
    print(f'Could not delete bucket: {e}')

### AWS resource cleanup

Payment sessions expire automatically when `expiryTimeInMinutes` elapses — no manual
deletion required. To clean up the payment manager and payment instrument, refer to
the cleanup section of the setup tutorial (`00-getting-started/00-setup-agentcore-payments/`).

---

## Shared Responsibility

| Responsibility | AWS | You |
|:---|:---:|:---:|
| Securing the AgentCore payments service infrastructure | ✅ | |
| Encrypting payment credentials at rest | ✅ | |
| Enforcing payment session limits at the service level | ✅ | |
| Settling on-chain transactions (Coinbase CDP) | ✅ | |
| Managing Runtime container compute and networking | ✅ | |
| Configuring IAM roles with least-privilege permissions | | ✅ |
| Setting appropriate `maxSpendAmount` payment limits | | ✅ |
| Protecting AWS credentials from exposure | | ✅ |
| Funding the payment instrument with sufficient USDC | | ✅ |
| Monitoring agent spend and session usage | | ✅ |
| Validating prompts to prevent prompt injection | | ✅ |
| Reviewing Heurist endpoint terms of service | | ✅ |

> **Security note:** Never commit `.env` files or payment credentials to source control.
> Use AWS Secrets Manager for production credentials.
> Payment sessions are time-bounded and budget-capped — set conservative `maxSpendAmount`
> limits when running agents in automated or unattended contexts.